# Baseline robustness: naive + val_mean (expected-zero-variance check)

Both models are fully deterministic (no fitting, no randomness) -- run twice per dataset anyway for structural consistency; expect std dev = 0.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import pm4py

ROOT = Path.cwd().resolve().parent.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "analysis"))
sys.path.insert(0, str(ROOT / "steady_state_detection"))

from ts_comparison import load_splits
from ts_comparison import STAT_FORECASTERS
from sklearn.metrics import mean_absolute_error, mean_squared_error

RESULTS = ROOT / "results"
ROBUST = ROOT / "robustness" / "baseline"

SYNTH_DATASETS = [p.stem for p in sorted((ROOT / "data" / "synthetic").glob("*.xes")) if "recency" not in p.stem]
REAL_DATASETS = ["bpic12-a", "bpic15-1", "bpic15-2", "bpic17-o",
                 "bpic20-dom", "bpic20-int", "helpdesk", "sepsis"]
SERIES = ["concurrent_cases", "throughput_time"]

NEW_SEEDS = [43, 44]


def forecast_val_mean(train: pd.Series, horizon: int, params=None) -> np.ndarray:
    """Constant forecast at the mean of the passed-in series."""
    return np.full(horizon, float(train.mean()))


print(f"{len(SYNTH_DATASETS)} synthetic datasets, {len(REAL_DATASETS)} real-life (ssd) datasets, seeds={NEW_SEEDS}")
print("NOTE: naive and val_mean are both fully deterministic -- seed sensitivity is")
print("non-relevant here. Run twice per dataset anyway for structural consistency with the other")
print("5 robustness notebooks; expect std dev = 0 (the correct, honest result, not a bug).")

In [ ]:
def run_baseline_robustness_one(dataset: str, is_real: bool, seed: int):
    """naive + val_mean, both deterministic -- seed is a label only. Fully
    resumable: skips if metrics already exist."""
    sub = "ssd" if is_real else "synthetic"
    out_dir = ROBUST / sub / dataset / f"seed_{seed}"
    metrics_path = out_dir / "metrics.csv"
    if metrics_path.exists():
        print(f"  [skip] {dataset}/seed_{seed}: metrics already exist")
        return

    if is_real:
        xes_path = ROOT / "data" / "real-life" / f"{dataset}.xes"
        _, cc, tt, _, _, _ = load_data_for_ssd(xes_path)
        splits = {"concurrent_cases": cc, "throughput_time": tt}
    else:
        split = load_splits(dataset, "none", is_real=False)
        splits = {"concurrent_cases": split["cc"], "throughput_time": split["tt"]}

    rows = []
    for series_name in SERIES:
        s = splits[series_name]
        train_val = pd.concat([s["train"], s["val"]])
        test_idx = s["test"].index
        actual = s["test"].to_numpy()
        if len(actual) == 0:
            continue
        for model_name, forecaster, series_in in [
            ("naive", STAT_FORECASTERS["naive"], train_val),
            ("val_mean", forecast_val_mean, s["val"]),
        ]:
            pred = forecaster(series_in, len(test_idx), {})
            rows.append(dict(dataset=dataset, series=series_name, model=model_name, seed=seed,
                             mae=mean_absolute_error(actual, pred), mse=mean_squared_error(actual, pred)))

    if not rows:
        print(f"  [skip] {dataset}/seed_{seed}: no rows produced (test set empty?)")
        return

    out_dir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(metrics_path, index=False)
    print(f"  [done] {dataset}/seed_{seed}: {len(rows)} (model, series) rows")

## Part 1: Synthetic

In [ ]:
for name in SYNTH_DATASETS:
    print(f"\n{'='*60}\n{name} (synthetic)\n{'='*60}")
    for seed in NEW_SEEDS:
        run_baseline_robustness_one(name, is_real=False, seed=seed)

## Part 2: SSD

In [ ]:
def load_data_for_ssd(xes_path):
    """Returns (df, cc, tt, train_df, val_df, test_df) for the ssd trim."""
    from time_series_preprocessing import Split3WayConfig, split_timeseries
    from time_series_creation import create_concurrent_cases_timeseries, create_avg_throughtput_time_timeseries
    from create_prefixes_from_windows import make_three_way_split
    from ssd_trim import run_ssd_trim

    log = pm4py.read_xes(str(xes_path))

    full_cc_raw = create_concurrent_cases_timeseries(log, plot=False)
    ssd_result = run_ssd_trim(log, window_step="D")
    canonical_end = ssd_result["cutoff"] if ssd_result["cutoff"] is not None else full_cc_raw.index[-1]
    full_cc_trimmed = full_cc_raw[full_cc_raw.index <= canonical_end]
    split_cfg = Split3WayConfig(train_frac=0.70, val_frac=0.10, test_frac=0.20)
    _, _, _, train_split, val_split = split_timeseries(full_cc_trimmed, split_cfg)

    full_tt_raw = create_avg_throughtput_time_timeseries(log, plot=False)
    full_tt_trimmed = full_tt_raw[full_tt_raw.index <= canonical_end]

    def _slice(raw, trimmed):
        idx = trimmed.index
        lo = train_split.tz_convert(None) if idx.tz is None else train_split
        hi = val_split.tz_convert(None) if idx.tz is None else val_split
        return {
            "raw": raw, "trimmed": trimmed,
            "train": trimmed[idx <= lo],
            "val": trimmed[(idx > lo) & (idx <= hi)],
            "test": trimmed[idx > hi],
            "train_split": train_split, "val_split": val_split,
        }

    cc = _slice(full_cc_raw, full_cc_trimmed)
    tt = _slice(full_tt_raw, full_tt_trimmed)

    df = pm4py.convert_to_dataframe(log)
    df["time:timestamp"] = pd.to_datetime(df["time:timestamp"], utc=True)
    df = df.dropna(subset=["case:concept:name"])
    _cols = {"case:concept:name": "caseid", "concept:name": "task",
             "lifecycle:transition": "event_type", "time:timestamp": "end_timestamp"}
    _cols["org:resource" if "org:resource" in df.columns else "org:group"] = "user"
    df = df.rename(columns=_cols)
    df["task"] = df["task"].fillna("unk")
    df["user"] = df["user"].fillna("unk")

    train_, val_, test_ = make_three_way_split(
        df, case_col="caseid", time_col="end_timestamp",
        train_split=cc["train_split"], val_split=cc["val_split"], full_traces=True,
    )
    return df, cc, tt, train_, val_, test_

In [ ]:
for name in REAL_DATASETS:
    print(f"\n{'='*60}\n{name} (ssd)\n{'='*60}")
    for seed in NEW_SEEDS:
        run_baseline_robustness_one(name, is_real=True, seed=seed)